In [1]:
# ============================================================
# LEVIR-Ship + Sentinel-2 Mixed Fine-tuning
# 1:1 or 2:1
# ONE CELL
# ============================================================

!pip install -q -U ultralytics gdown pyyaml

import os
import shutil
import zipfile
import random
from pathlib import Path

import gdown
import yaml
import torch
from ultralytics import YOLO
from google.colab import drive


# ============================================================
# 1. Google Drive
# ============================================================

drive.mount("/content/drive")


# ============================================================
# 2. Paths
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/LEVIR_Ship_YOLO26"
)

SOURCE_MODEL = (
    PROJECT_DIR
    / "YOLO26s_LEVIR_Ship"
    / "weights"
    / "best.pt"
)

TARGET_DIR = (
    PROJECT_DIR
    / "Sentinel2_Ship_Dataset"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "Mixed_Finetune"
)

# LEVIR는 Colab local에 다시 다운로드
LOCAL_DIR = Path(
    "/content/LEVIR_Ship"
)

DOWNLOAD_DIR = (
    LOCAL_DIR
    / "downloads"
)

DATASET_DIR = (
    LOCAL_DIR
    / "dataset"
)

DOWNLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DATASET_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 3. Official LEVIR-Ship Google Drive IDs
# ============================================================

FILES = {
    "train": "1_kjTr4mpF1g2fWhAodKZXrsZjWrCfPh5",
    "val":   "1q2KFLVYU1SbeSU4Uksj6FV9C28YmUsb1",
    "test":  "1SRq7hq7glKzzePWqtSlt0C_RVtc1pQQC",
}


# ============================================================
# 4. Download LEVIR dataset
# ============================================================

print("=" * 70)
print("DOWNLOAD LEVIR-SHIP")
print("=" * 70)

for split, file_id in FILES.items():

    zip_path = (
        DOWNLOAD_DIR
        / f"{split}.zip"
    )

    if not zip_path.exists():

        print(
            f"\nDownloading {split}..."
        )

        url = (
            "https://drive.google.com/uc"
            f"?id={file_id}"
        )

        gdown.download(
            url,
            str(zip_path),
            quiet=False
        )

    else:

        print(
            f"{split}.zip already exists"
        )


# ============================================================
# 5. Extract LEVIR
# ============================================================

print()
print("=" * 70)
print("EXTRACT LEVIR-SHIP")
print("=" * 70)

for split in FILES.keys():

    zip_path = (
        DOWNLOAD_DIR
        / f"{split}.zip"
    )

    extract_dir = (
        DATASET_DIR
        / split
    )

    if not extract_dir.exists():

        extract_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        print(
            f"Extracting {split}..."
        )

        with zipfile.ZipFile(
            zip_path,
            "r"
        ) as z:

            z.extractall(
                extract_dir
            )

    else:

        print(
            f"{split} already extracted"
        )


# ============================================================
# 6. Image extensions
# ============================================================

IMG_EXTS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp"
}


def get_images(folder):

    return sorted([
        p
        for p in folder.rglob("*")
        if p.is_file()
        and p.suffix.lower() in IMG_EXTS
    ])


# ============================================================
# 7. Find LEVIR train/images automatically
# ============================================================

source_candidates = []

for p in DATASET_DIR.rglob("*"):

    if not p.is_dir():
        continue

    # pattern 1
    a_img = p / "images"
    a_lab = p / "labels"

    if (
        a_img.exists()
        and a_lab.exists()
    ):

        imgs = get_images(
            a_img
        )

        if len(imgs) > 0:

            source_candidates.append(
                (
                    p,
                    a_img,
                    a_lab
                )
            )


if len(source_candidates) == 0:

    raise FileNotFoundError(
        "LEVIR images/labels 폴더를 찾지 못했습니다."
    )


# 가장 이미지 많은 폴더를 train으로 선택
source_candidates.sort(
    key=lambda x: len(
        get_images(x[1])
    ),
    reverse=True
)

SOURCE_ROOT, SOURCE_IMG, SOURCE_LAB = (
    source_candidates[0]
)


print()
print("=" * 70)
print("LEVIR SOURCE")
print("=" * 70)

print(
    "Root:",
    SOURCE_ROOT
)

print(
    "Images:",
    SOURCE_IMG
)

print(
    "Labels:",
    SOURCE_LAB
)


# ============================================================
# 8. Sentinel-2 paths
# ============================================================

TARGET_TRAIN_IMG = (
    TARGET_DIR
    / "train"
    / "images"
)

TARGET_TRAIN_LAB = (
    TARGET_DIR
    / "train"
    / "labels"
)

TARGET_VAL_IMG = (
    TARGET_DIR
    / "valid"
    / "images"
)

TARGET_TEST_IMG = (
    TARGET_DIR
    / "test"
    / "images"
)


for p in [
    TARGET_TRAIN_IMG,
    TARGET_TRAIN_LAB,
    TARGET_VAL_IMG
]:

    assert p.exists(), (
        f"Missing: {p}"
    )


assert SOURCE_MODEL.exists(), (
    f"best.pt missing:\n{SOURCE_MODEL}"
)


# ============================================================
# 9. Mixing ratio
# ============================================================
#
# 1 = LEVIR : Sentinel-2 = 1 : 1
# 2 = LEVIR : Sentinel-2 = 2 : 1
# ============================================================

SOURCE_TARGET_RATIO = 5

SEED = 42

random.seed(
    SEED
)


# ============================================================
# 10. Collect images
# ============================================================

source_images = get_images(
    SOURCE_IMG
)

target_images = get_images(
    TARGET_TRAIN_IMG
)


print()
print("=" * 70)
print("DATA SIZE")
print("=" * 70)

print(
    "LEVIR:",
    len(source_images)
)

print(
    "Sentinel-2:",
    len(target_images)
)


# ============================================================
# 11. Sample source images
# ============================================================

n_target = len(
    target_images
)

n_source = (
    n_target
    * SOURCE_TARGET_RATIO
)


if n_source <= len(source_images):

    selected_source = random.sample(
        source_images,
        n_source
    )

else:

    selected_source = random.choices(
        source_images,
        k=n_source
    )


mixed_images = (
    selected_source
    + target_images
)

random.shuffle(
    mixed_images
)


print()
print(
    f"Mix ratio "
    f"LEVIR:Sentinel = "
    f"{SOURCE_TARGET_RATIO}:1"
)

print(
    "LEVIR selected:",
    len(selected_source)
)

print(
    "Sentinel:",
    len(target_images)
)

print(
    "Total:",
    len(mixed_images)
)


# ============================================================
# 12. Create train.txt
# ============================================================

MIX_DIR = (
    OUTPUT_DIR
    / f"ratio_{SOURCE_TARGET_RATIO}_to_1"
)

MIX_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TRAIN_TXT = (
    MIX_DIR
    / "mixed_train.txt"
)


with open(
    TRAIN_TXT,
    "w"
) as f:

    for img in mixed_images:

        f.write(
            str(img.resolve())
            + "\n"
        )


# ============================================================
# 13. Create YAML
# ============================================================

DATA_YAML = (
    MIX_DIR
    / "mixed_data.yaml"
)


yaml_data = {

    "train": str(
        TRAIN_TXT
    ),

    # validation은 Sentinel-2 only
    "val": str(
        TARGET_VAL_IMG
    ),

    "test": str(
        TARGET_TEST_IMG
    ),

    "nc": 1,

    "names": {
        0: "ship"
    }
}


with open(
    DATA_YAML,
    "w"
) as f:

    yaml.safe_dump(
        yaml_data,
        f,
        sort_keys=False
    )


print()
print("=" * 70)
print("YAML")
print("=" * 70)

print(
    DATA_YAML.read_text()
)


# ============================================================
# 14. GPU
# ============================================================

DEVICE = (
    0
    if torch.cuda.is_available()
    else "cpu"
)

print()
print(
    "Device:",
    DEVICE
)

if torch.cuda.is_available():

    print(
        torch.cuda.get_device_name(0)
    )


# ============================================================
# 15. Load ORIGINAL LEVIR best.pt
# ============================================================

model = YOLO(
    str(SOURCE_MODEL)
)


# ============================================================
# 16. Mixed Fine-tuning
# ============================================================

RUN_NAME = (
    f"LEVIR_Sentinel2_"
    f"{SOURCE_TARGET_RATIO}to1_30epoch" #epoch에 맞추어 수정
)


print()
print("=" * 70)
print("START TRAINING")
print("=" * 70)


results = model.train(

    data=str(DATA_YAML),

    epochs=30,

    imgsz=640,

    batch=16,

    device=DEVICE,

    workers=2,

    optimizer="AdamW",

    lr0=0.001,

    lrf=0.01,

    weight_decay=0.0005,

    patience=10,

    project=str(
        OUTPUT_DIR
    ),

    name=RUN_NAME,

    exist_ok=True,

    plots=True,

    seed=SEED
)


# ============================================================
# 17. Best model
# ============================================================

BEST_MODEL = (
    OUTPUT_DIR
    / RUN_NAME
    / "weights"
    / "best.pt"
)


print()
print("=" * 70)
print("FINISHED")
print("=" * 70)

print(
    "Best model:"
)

print(
    BEST_MODEL
)

print(
    "Exists:",
    BEST_MODEL.exists()
)


# ============================================================
# 18. Sentinel-2 validation
# ============================================================

if BEST_MODEL.exists():

    ft_model = YOLO(
        str(BEST_MODEL)
    )

    metrics = ft_model.val(

        data=str(DATA_YAML),

        split="val",

        imgsz=640,

        batch=16,

        device=DEVICE,

        plots=True
    )

    print()
    print("=" * 70)
    print("SENTINEL-2 RESULTS")
    print("=" * 70)

    try:

        print(
            "Precision:",
            metrics.box.mp
        )

        print(
            "Recall:",
            metrics.box.mr
        )

        print(
            "mAP50:",
            metrics.box.map50
        )

        print(
            "mAP50-95:",
            metrics.box.map
        )

    except:
        pass

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Mounted at /content/drive
DOWNLOAD LEVIR-SHIP



Downloading...
From (original): https://drive.google.com/uc?id=1_kjTr4mpF1g2fWhAodKZXrsZjWrCfPh5
From (redirected): https://drive.google.com/uc?id=1_kjTr4mpF1g2fWhAodKZXrsZjWrCfPh5&confirm=t&uuid=080ca140-f036-497e-9335-5a366584ab3d
To: /content/LEVIR_Ship/downloads/train.zip
100%|██████████| 632M/632M [00:06<00:00, 95.4MB/s]


Downloading...
From (original): https://drive.google.com/uc?id=1q2KFLVYU1SbeSU4Uksj6FV9C28YmUsb1
From (redirected): https://drive.google.com/uc?id=1q2KFLVYU1SbeSU4Uksj6FV9C28YmUsb1&confirm=t&uuid=6f307cc9-e544-4caa-baae-68258ce0d9f2
To: /content/LEVIR_Ship/downloads/val.zip
100%|██████████| 216M/216M [00:01<00:00, 112MB/s] 


Downloading...
From (original): https://drive.google.com/uc?id=1SRq7hq7glKzzePWqtSlt0C_RVtc1pQQC
From (redirected): https://drive.google.com/uc?id=1SRq7hq7glKzzePWqtSlt0C_RVtc1pQQC&confirm=t&uuid=0087f2f6-93cf-41a5-a407-5ab841414901
To: /content/LEVIR_Ship/downloads/test.zip
100%|██████████| 207M/207M [00:04<00:00, 48.3MB/s]



EXTRACT LEVIR-SHIP
Extracting train...
Extracting val...
Extracting test...

LEVIR SOURCE
Root: /content/LEVIR_Ship/dataset/train/train
Images: /content/LEVIR_Ship/dataset/train/train/images
Labels: /content/LEVIR_Ship/dataset/train/train/labels

DATA SIZE
LEVIR: 2320
Sentinel-2: 582

Mix ratio LEVIR:Sentinel = 5:1
LEVIR selected: 2910
Sentinel: 582
Total: 3492

YAML
train: /content/drive/MyDrive/LEVIR_Ship_YOLO26/Mixed_Finetune/ratio_5_to_1/mixed_train.txt
val: /content/drive/MyDrive/LEVIR_Ship_YOLO26/Sentinel2_Ship_Dataset/valid/images
test: /content/drive/MyDrive/LEVIR_Ship_YOLO26/Sentinel2_Ship_Dataset/test/images
nc: 1
names:
  0: ship


Device: 0
Tesla T4

START TRAINING
Ultralytics 8.4.142 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_rema